In [ ]:
import os
from typing import Annotated, TypedDict, List
from langgraph.graph import StateGraph, START, END
from langchain_huggingface import HuggingFaceEndpoint
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate,  MessagesPlaceholder
from langgraph.graph.message import add_messages
from langchain_community.tools.tavily_search import TavilySearchResults
from nemoguardrails import LLMRails, RailsConfig
from contextlib import asynccontextmanager
from psycopg_pool import AsyncConnectionPool
from langgraph.checkpoint.postgres.aio import AsyncPostgresSaver
from fastapi import FastAPI
from fastapi.responses import StreamingResponse
import asyncio
from db_config import get_vector_db

In [ ]:

# --- 1. LLM CONFIGURATION ---
llm = HuggingFaceEndpoint(
    repo_id="meta-llama/Llama-3.1-8B-Instruct",
    huggingfacehub_api_token=os.getenv("HF_TOKEN"),
    temperature=0.1,
    max_new_tokens=512,
    streaming=True
)

retriever = None

In [ ]:
# --- 3. STATE DEFINITION ---
class AgentState(TypedDict):
    messages: Annotated[List[BaseMessage], add_messages]
    context: str
    status: str

In [ ]:
# --- 4. NODES / AGENT LOGIC ---


# 1). INITIALIZE GUARDRAILS ---
config = RailsConfig.from_path("./config")
rails = LLMRails(config, llm=llm)  # Pass existing LLM


# 2). RETRIEVE NODE ---

search_tool = TavilySearchResults(k=3)


async def retrieve_node(state: AgentState):
    """Fetch relevant documents + web search based on the last user query."""
    last_message = state["messages"][-1].content

    web_task = search_tool.ainvoke(last_message)
    pdf_task = retriever.ainvoke(last_message)

    web_results, docs = await asyncio.gather(web_task, pdf_task, return_exceptions=True)

    
    pdf_context = ""
    web_context = ""

    if not isinstance(docs, Exception):
        pdf_context = "\n".join([doc.page_content for doc in docs])

    if not isinstance(web_results, Exception):
        web_context = "\n".join([r["content"] for r in web_results])


    combined_context = (
        f"--- INTERNAL PDF KNOWLEDGE ---\n{pdf_context}\n\n"
        f"--- LIVE WEB DATA ---\n{web_context}"
    )

    return {
        "context": combined_context,
        "status": "Scanning PDFs and searching the web...",
    }
#  3). CALL MODEL NODE WITH GUARDRAILS & PROMPT TEMPLATE ---
async def call_model_node(state: AgentState):
    context = state["context"]
    
    # A. Define the Prompt Template
    prompt_template = ChatPromptTemplate.from_messages(
        [
            (
                "system",
                "Answer concisely based ONLY on the provided context. "
                "If the answer is not in the context, say you don't know.\n\n"
                "Context:\n{context}",
            ),
            MessagesPlaceholder(variable_name="history"),
            ("human", "{question}"),
        ]
    )

    # B. Format the template into a list of BaseMessages
    # This automatically handles variable injection for history and the current question
    formatted_messages = prompt_template.invoke({
        "context": context,
        "history": state["messages"][:-1],  # All previous messages
        "question": state["messages"][-1].content  # The latest user query
    }).to_messages()

    # C. Convert LangChain BaseMessages to NeMo Guardrails dictionary format
    # This preserves the structured message rendering from LangChain
    nemo_messages = []
    
    for msg in formatted_messages:
        if msg.type == "system":
            role = "system"
        elif msg.type == "human":
            role = "user"
        else:
            role = "assistant"
        nemo_messages.append({"role": role, "content": msg.content})

    # D. Generate response with Guardrails
    res = await rails.generate_async(messages=nemo_messages)
    response_content = res.content
    status = "Finalizing response..."
    
    if not response_content or "I cannot answer" in response_content:
        status = "Response blocked by safety/fact-check guardrails."

    return {"messages": [AIMessage(content=response_content)], "status": status}

In [ ]:
# --- 5. GRAPH ORCHESTRATION ---

workflow = StateGraph(AgentState)

workflow.add_node("retrieve", retrieve_node)
workflow.add_node("llm", call_model_node)

workflow.add_edge(START, "retrieve")
workflow.add_edge("retrieve", "llm")
workflow.add_edge("llm", END)

In [ ]:
# --- 6. MEMORY (PostgreSQL Persistence) ---

DB_USER = os.environ.get("PGSQL_USERNAME")
DB_PASSWORD = os.environ.get("PGSQL_PASSWORD")
DB_HOST = os.environ.get("PGSQL_HOST", "localhost")
DB_PORT = os.environ.get("PGSQL_PORT", "5432")
DB_NAME = os.environ.get("PGSQL_NAME")

DB_URI = (
    f"postgresql://{DB_USER}:{DB_PASSWORD}"
    f"@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

pool = AsyncConnectionPool(conninfo=DB_URI, max_size=10, kwargs={"autocommit": True})


@asynccontextmanager
async def lifespan(app: FastAPI):
    global retriever
    global graph_app
    # Initialize checkpointer and setup tables
    async with pool:
        vector_db = get_vector_db()
        retriever = vector_db.as_retriever(search_kwargs={"k": 3})
        checkpointer = AsyncPostgresSaver(pool)
        await checkpointer.setup()
        # Compile graph with the async checkpointer
        graph_app = workflow.compile(checkpointer=checkpointer)
        yield

In [ ]:
# --- 7. API / FRONTEND CONNECTION (FastAPI) ---

api = FastAPI(lifespan=lifespan)

@api.post("/chat")
async def chat_endpoint(user_id: str, thread_id: str, message: str):
    config = {"configurable": {"thread_id": thread_id}}
    input_data = {"messages": [HumanMessage(content=message)]}
    
    async def event_generator():
        async for event in graph_app.astream(input_data, config=config, stream_mode="updates"):
            # 1. Handle Status Updates (from any node that provides them)
            # The 'event' dict will look like: {"retrieve": {"status": "...", "context": "..."}}
            node_name = list(event.keys())[0]
            node_output = event[node_name]

            if "status" in node_output:
                yield f"data: [STATUS] {node_output['status']}\n\n"

            # 2. Handle the Final AI Message (specifically from the llm node)
            if node_name == "llm" and "messages" in node_output:
                # node_output["messages"] only contains the NEW messages from this node
                final_content = node_output["messages"][-1].content
                yield f"data: {final_content}\n\n"


    return StreamingResponse(event_generator(), media_type="text/event-stream")